In [5]:
## Part 1: Set Up the "Slow" Environment (10 min)
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
import random
import time

# Stop the existing Spark session if it exists
if 'spark' in globals() and spark.sparkContext._jsc is not None:
    spark.stop()
    print("Spark session stopped.")

# ANTI-PATTERN: broadcast disabled, too many shuffle partitions
spark = SparkSession.builder \
    .appName("StreamPulse-Revenue-SLOW") \
    .master("local[*]") \
    .config("spark.driver.memory", "2g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.sql.adaptive.enabled", "false") \
    .config("spark.sql.autoBroadcastJoinThreshold", "-1") \
    .getOrCreate()

print("✅ SparkSession created (intentionally misconfigured)")

Spark session stopped.
✅ SparkSession created (intentionally misconfigured)


In [6]:
## Generate the revenue dataset:
random.seed(42)
N = 600000

# Events (large)
event_data = []
for i in range(N):
    event_data.append((
        f"EVT-{i+1:07d}",
        f"USR-{random.randint(1, 100000):06d}",
        f"ART-{random.randint(1, 5000):05d}",
        random.choice(["Pop", "Rock", "Hip-Hop", "Jazz", "Electronic", "R&B", "Country", "Classical"]),
        random.choice(["North America", "Europe", "Asia Pacific", "Latin America", "Africa"]),
        random.randint(15, 350),
        random.choice([True, False]),
        random.choice(["mobile", "desktop", "smart_speaker", "tablet", "car", "tv"]),
        f"2024-{random.randint(1,12):02d}-{random.randint(1,28):02d}",
    ))

events = spark.createDataFrame(event_data,
    ["event_id", "user_id", "artist_id", "genre", "region",
     "duration_sec", "completed", "device", "event_date"]) \
    .withColumn("event_date", col("event_date").cast("date")) \
    .withColumn("month", month(col("event_date")))

events.write.parquet("revenue_data/events", mode="overwrite")

# Subscriptions (medium - 100K users with subscription info)
sub_data = [(f"USR-{i+1:06d}",
             random.choice(["free", "individual", "family", "student"]),
             __builtins__.round(random.choice([0.0, 9.99, 14.99, 4.99]), 2),
             random.choice(["US", "UK", "DE", "JP", "BR", "IN", "KR", "FR"]))
            for i in range(100000)]
subscriptions = spark.createDataFrame(sub_data, ["user_id", "plan", "monthly_price", "country"])
subscriptions.write.parquet("revenue_data/subscriptions", mode="overwrite")

# Ad rates (tiny - 8 genres x 6 devices = 48 rows)
ad_data = []
for genre in ["Pop", "Rock", "Hip-Hop", "Jazz", "Electronic", "R&B", "Country", "Classical"]:
    for device in ["mobile", "desktop", "smart_speaker", "tablet", "car", "tv"]:
        cpm = __builtins__.round(random.uniform(1.5, 8.0), 2)
        ad_data.append((genre, device, cpm))
ad_rates = spark.createDataFrame(ad_data, ["ad_genre", "ad_device", "cpm"])
ad_rates.write.parquet("revenue_data/ad_rates", mode="overwrite")

# Artist payout rates (small - 5000 artists)
payout_data = [(f"ART-{i+1:05d}", __builtins__.round(random.uniform(0.003, 0.008), 4),
                random.choice(["major", "indie", "unsigned"]))
               for i in range(5000)]
payouts = spark.createDataFrame(payout_data, ["artist_id", "per_stream_rate", "label_type"])
payouts.write.parquet("revenue_data/payouts", mode="overwrite")

# Reload from disk
events = spark.read.parquet("revenue_data/events")
subscriptions = spark.read.parquet("revenue_data/subscriptions")
ad_rates = spark.read.parquet("revenue_data/ad_rates")
payouts = spark.read.parquet("revenue_data/payouts")

print(f"Events: {events.count()} | Subs: {subscriptions.count()} | "
      f"Ad rates: {ad_rates.count()} | Payouts: {payouts.count()}")


Events: 600000 | Subs: 100000 | Ad rates: 48 | Payouts: 5000


In [7]:
## Part 2: The Unoptimized Pipeline (Baseline) (15 min)
## Run the slow pipeline and time it:
from pyspark.sql.functions import countDistinct

print("=" * 60)
print("RUNNING UNOPTIMIZED PIPELINE (BASELINE)")
print("=" * 60)

total_start = time.time()

# Build enriched revenue DataFrame (NOT cached, recomputed every time)
def build_revenue():
    return events \
        .join(subscriptions, "user_id") \
        .join(ad_rates,
              (events.genre == ad_rates.ad_genre) & (events.device == ad_rates.ad_device)) \
        .join(payouts, "artist_id") \
        .withColumn("ad_revenue", col("cpm") / 1000) \
        .withColumn("stream_payout", col("per_stream_rate")) \
        .withColumn("is_premium", when(col("plan") != "free", True).otherwise(False))

# Report 1: Genre Revenue
revenue = build_revenue()
r1_start = time.time()
report_1 = revenue.groupBy("genre") \
    .agg(sum("ad_revenue").alias("total_ad_rev"),
         countDistinct("user_id").alias("unique_listeners")) \
    .collect()
r1_time = time.time() - r1_start

# Report 2: Regional Breakdown
revenue = build_revenue()
r2_start = time.time()
report_2 = revenue.groupBy("region", "country") \
    .agg(count("*").alias("streams"),
         sum("ad_revenue").alias("ad_rev")) \
    .collect()
r2_time = time.time() - r2_start

# Report 3: Subscription Analysis
revenue = build_revenue()
r3_start = time.time()
report_3 = revenue.groupBy("plan") \
    .agg(countDistinct("user_id").alias("users"),
         count("*").alias("total_streams"),
         avg("duration_sec").alias("avg_duration")) \
    .collect()
r3_time = time.time() - r3_start

# Report 4: Ad Performance
revenue = build_revenue()
r4_start = time.time()
report_4 = revenue.groupBy("device", "genre") \
    .agg(sum("ad_revenue").alias("total_ad_rev"),
         count("*").alias("impressions")) \
    .collect()
r4_time = time.time() - r4_start

# Report 5: Artist Payouts
revenue = build_revenue()
r5_start = time.time()
report_5 = revenue.groupBy("artist_id", "label_type") \
    .agg(sum("stream_payout").alias("total_payout"),
         count("*").alias("total_streams")) \
    .orderBy(desc("total_payout")).limit(100) \
    .collect()
r5_time = time.time() - r5_start

# Report 6: Daily Summary
revenue = build_revenue()
r6_start = time.time()
report_6 = revenue.groupBy("event_date") \
    .agg(count("*").alias("streams"),
         sum("ad_revenue").alias("ad_rev"),
         countDistinct("user_id").alias("unique_users")) \
    .orderBy("event_date") \
    .collect()
r6_time = time.time() - r6_start

baseline_total = time.time() - total_start

print(f"\nReport 1 (genre):        {r1_time:.2f}s")
print(f"Report 2 (regional):     {r2_time:.2f}s")
print(f"Report 3 (subscription): {r3_time:.2f}s")
print(f"Report 4 (ad perf):      {r4_time:.2f}s")
print(f"Report 5 (payouts):      {r5_time:.2f}s")
print(f"Report 6 (daily):        {r6_time:.2f}s")
print(f"\n⏱️  BASELINE TOTAL: {baseline_total:.2f}s")


RUNNING UNOPTIMIZED PIPELINE (BASELINE)

Report 1 (genre):        57.36s
Report 2 (regional):     24.64s
Report 3 (subscription): 36.85s
Report 4 (ad perf):      23.51s
Report 5 (payouts):      15.33s
Report 6 (daily):        41.47s

⏱️  BASELINE TOTAL: 200.87s


In [8]:
## Analyze the baseline plan:
print("\nBASELINE PLAN:")
build_revenue().groupBy("genre").agg(sum("ad_revenue")).explain(mode="formatted")



BASELINE PLAN:
== Physical Plan ==
* HashAggregate (33)
+- Exchange (32)
   +- * HashAggregate (31)
      +- * Project (30)
         +- * SortMergeJoin Inner (29)
            :- * Sort (23)
            :  +- Exchange (22)
            :     +- * Project (21)
            :        +- * SortMergeJoin Inner (20)
            :           :- * Sort (14)
            :           :  +- Exchange (13)
            :           :     +- * Project (12)
            :           :        +- * SortMergeJoin Inner (11)
            :           :           :- * Sort (5)
            :           :           :  +- Exchange (4)
            :           :           :     +- * Filter (3)
            :           :           :        +- * ColumnarToRow (2)
            :           :           :           +- Scan parquet  (1)
            :           :           +- * Sort (10)
            :           :              +- Exchange (9)
            :           :                 +- * Filter (8)
            :           :       

In [9]:
## Document the anti-patterns you find:
Anti-Pattern | Description | Impact
-------------|-------------|--------
Broadcast join disabled | `spark.sql.autoBroadcastJoinThreshold` set to `-1`, forcing SortMergeJoins even for tiny tables (ad_rates with 48 rows, payouts with 5000 rows) | Massive shuffle overhead for small dimension tables that should be broadcast
No caching of reused DataFrame | `build_revenue()` called 6 times, recomputing the exact same 4-table join pipeline each time | 6x redundant processing of entire dataset
Disabled Adaptive Query Execution | `spark.sql.adaptive.enabled` set to `false` | No dynamic optimization, coalescing, or join strategy adjustments
Excessive shuffle partitions | `spark.sql.shuffle.partitions` set to 200 regardless of data size | 200 small files/tasks for tiny aggregations, scheduler overhead
No column pruning | Each report selects all columns when only a subset needed | Shuffles and reads unnecessary data across all stages

SyntaxError: invalid decimal literal (ipython-input-2726755225.py, line 5)

In [10]:
## Optimization 1: Enable Broadcast for Small Tables
# Enable broadcast join threshold (10MB default)
# Reset config
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
import time

# Create SparkSession with baseline config first
spark = SparkSession.builder \
    .appName("StreamPulse-Revenue") \
    .master("local[*]") \
    .config("spark.driver.memory", "2g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.sql.adaptive.enabled", "false") \
    .config("spark.sql.autoBroadcastJoinThreshold", "-1") \
    .getOrCreate()

print("✅ SparkSession created")

# Now load your data
events = spark.read.parquet("revenue_data/events")
subscriptions = spark.read.parquet("revenue_data/subscriptions")
ad_rates = spark.read.parquet("revenue_data/ad_rates")
payouts = spark.read.parquet("revenue_data/payouts")

print(f"Events: {events.count()} | Subs: {subscriptions.count()} | "
      f"Ad rates: {ad_rates.count()} | Payouts: {payouts.count()}")

# Now you can run the baseline timing
print("=" * 60)
print("RUNNING UNOPTIMIZED PIPELINE (BASELINE)")
print("=" * 60)

total_start = time.time()

✅ SparkSession created
Events: 600000 | Subs: 100000 | Ad rates: 48 | Payouts: 5000
RUNNING UNOPTIMIZED PIPELINE (BASELINE)


In [12]:
## Optimization 2: Cache the Enriched DataFrame
# Build enriched DataFrame ONCE
revenue_opt = events \
    .join(subscriptions, "user_id") \
    .join(broadcast(ad_rates),
          (events.genre == ad_rates.ad_genre) & (events.device == ad_rates.ad_device)) \
    .join(broadcast(payouts), "artist_id") \
    .withColumn("ad_revenue", col("cpm") / 1000) \
    .withColumn("stream_payout", col("per_stream_rate"))

# Cache it!
revenue_opt.cache()

# Force cache (materialize)
print(f"Caching {revenue_opt.count():,} rows...")

# Now run ALL 6 reports from cache without recomputation
report_1 = revenue_opt.groupBy("genre").agg(sum("ad_revenue")).collect()  # Uses cache
report_2 = revenue_opt.groupBy("region", "country").agg(count("*")).collect()  # Uses cache
# ... etc

# Clean up when done
# revenue_opt.unpersist()

# EXPLANATION:
# 1. First query materializes the cache (takes time)
# 2. Subsequent 5 queries read from memory instead of recomputing
# 3. Eliminates 5 full pipeline executions
# 4. Memory usage ~ size of enriched DataFrame

Caching 600,000 rows...


In [13]:
### Optimization 3: Reduce Shuffle Partitions
# Before optimization
print(f"Current shuffle partitions: {spark.conf.get('spark.sql.shuffle.partitions')}")  # 200

# Reduce to match cluster size (8 for local mode)
spark.conf.set("spark.sql.shuffle.partitions", "8")

# Verify change
print(f"New shuffle partitions: {spark.conf.get('spark.sql.shuffle.partitions')}")  # 8

# EXPLANATION:
# 1. 200 partitions creates 200 small files/tasks for tiny aggregations
# 2. 8 partitions matches local cores and data volume
# 3. Reduces scheduler overhead and file I/O
# 4. Each aggregation now uses 8 partitions instead of 200

Current shuffle partitions: 200
New shuffle partitions: 8


In [14]:
## Optimization 4: Column Pruning
# BEFORE: Reading all columns from each table
events_all = events  # Reads all 10 columns
subscriptions_all = subscriptions  # Reads all 4 columns

# AFTER: Select only needed columns for reports
events_pruned = events.select(
    "event_id", "user_id", "artist_id", "genre", "region",
    "duration_sec", "device", "event_date"  # Only 8 of 10 columns
)

subscriptions_pruned = subscriptions.select(
    "user_id", "plan", "country"  # Only 3 of 4 columns (drop monthly_price)
)

ad_rates_pruned = ad_rates.select(
    "ad_genre", "ad_device", "cpm"  # All 3 columns needed
)

payouts_pruned = payouts.select(
    "artist_id", "per_stream_rate", "label_type"  # All 3 columns needed
)

# Build optimized pipeline with pruned columns
revenue_opt = events_pruned \
    .join(subscriptions_pruned, "user_id") \
    .join(broadcast(ad_rates_pruned),
          (col("genre") == col("ad_genre")) & (col("device") == col("ad_device"))) \
    .join(broadcast(payouts_pruned), "artist_id") \
    .withColumn("ad_revenue", col("cpm") / 1000) \
    .withColumn("stream_payout", col("per_stream_rate"))

# EXPLANATION:
# 1. Each report uses different columns, but all share core set
# 2. Pruning reduces data read from disk and shuffled
# 3. Especially important for large events table
# 4. Drops unused columns like 'month', 'monthly_price'

In [19]:
## Optimization 5: Filter Early
# IF reports only need specific time periods (e.g., Q1 2024)
revenue_opt = events.filter(
    (col("event_date") >= "2024-01-01") &
    (col("event_date") <= "2024-03-31")
).join(
    subscriptions.select("user_id", "plan", "country"), "user_id"
).join(
    broadcast(ad_rates),
    (col("genre") == col("ad_genre")) & (col("device") == col("ad_device"))
).join(
    broadcast(payouts), "artist_id"
).withColumn(
    "ad_revenue", col("cpm") / 1000
).withColumn(
    "stream_payout", col("per_stream_rate")
)

# IF reports need specific genres only
revenue_opt = events.filter(
    col("genre").isin(["Pop", "Rock", "Electronic"])
).join(
    subscriptions.select("user_id", "plan", "country"), "user_id"
).join(
    broadcast(ad_rates),
    (col("genre") == col("ad_genre")) & (col("device") == col("ad_device"))
).join(
    broadcast(payouts), "artist_id"
).withColumn(
    "ad_revenue", col("cpm") / 1000
).withColumn(
    "stream_payout", col("per_stream_rate")
)

# EXPLANATION:
# 1. Filtering before joins reduces data processed in all subsequent stages
# 2. Especially effective if reports only need subset of data
# 3. Enables partition pruning if filtering on partition columns
# 4. Compound effect with other optimizations

In [20]:
## Combine ALL optimizations into a production-ready pipeline:
print("=" * 60)
print("RUNNING FULLY OPTIMIZED PIPELINE")
print("=" * 60)

# Reset config
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "10485760")
spark.conf.set("spark.sql.shuffle.partitions", "8")

total_start = time.time()

# Build enriched DataFrame ONCE with all optimizations
revenue_opt = events \
    .select("event_id", "user_id", "artist_id", "genre", "region",
            "duration_sec", "completed", "device", "event_date", "month") \
    .join(subscriptions.select("user_id", "plan", "country"), "user_id") \
    .join(broadcast(ad_rates),
          (col("genre") == col("ad_genre")) & (col("device") == col("ad_device"))) \
    .join(broadcast(payouts), "artist_id") \
    .withColumn("ad_revenue", col("cpm") / 1000) \
    .withColumn("stream_payout", col("per_stream_rate")) \
    .drop("ad_genre", "ad_device")

# Cache the shared DataFrame
revenue_opt.cache()
cache_start = time.time()
row_count = revenue_opt.count()
cache_time = time.time() - cache_start
print(f"✅ Cached {row_count} rows in {cache_time:.2f}s")

# Run all 6 reports from cache
r1 = revenue_opt.groupBy("genre").agg(sum("ad_revenue"), countDistinct("user_id")).collect()
r2 = revenue_opt.groupBy("region", "country").agg(count("*"), sum("ad_revenue")).collect()
r3 = revenue_opt.groupBy("plan").agg(countDistinct("user_id"), count("*"), avg("duration_sec")).collect()
r4 = revenue_opt.groupBy("device", "genre").agg(sum("ad_revenue"), count("*")).collect()
r5 = revenue_opt.groupBy("artist_id", "label_type") \
    .agg(sum("stream_payout"), count("*")) \
    .orderBy(desc("sum(stream_payout)")).limit(100).collect()
r6 = revenue_opt.groupBy("event_date") \
    .agg(count("*"), sum("ad_revenue"), countDistinct("user_id")) \
    .orderBy("event_date").collect()

optimized_total = time.time() - total_start

print(f"\n⏱️  OPTIMIZED TOTAL: {optimized_total:.2f}s")
print(f"⏱️  BASELINE TOTAL:  {baseline_total:.2f}s")
print(f"📈 SPEEDUP:          {baseline_total/optimized_total:.1f}x")
print(f"📉 TIME SAVED:       {baseline_total - optimized_total:.2f}s ({(1-optimized_total/baseline_total)*100:.0f}%)")

# Verify the plan
print("\nOPTIMIZED PLAN:")
revenue_opt.groupBy("genre").agg(sum("ad_revenue")).explain(mode="formatted")

revenue_opt.unpersist()


RUNNING FULLY OPTIMIZED PIPELINE
✅ Cached 600000 rows in 5.97s

⏱️  OPTIMIZED TOTAL: 18.30s
⏱️  BASELINE TOTAL:  200.87s
📈 SPEEDUP:          11.0x
📉 TIME SAVED:       182.57s (91%)

OPTIMIZED PLAN:
== Physical Plan ==
* HashAggregate (26)
+- Exchange (25)
   +- * HashAggregate (24)
      +- InMemoryTableScan (1)
            +- InMemoryRelation (2)
                  +- * Project (23)
                     +- * BroadcastHashJoin Inner BuildRight (22)
                        :- * Project (17)
                        :  +- * BroadcastHashJoin Inner BuildRight (16)
                        :     :- * Project (11)
                        :     :  +- * BroadcastHashJoin Inner BuildRight (10)
                        :     :     :- * Filter (5)
                        :     :     :  +- * ColumnarToRow (4)
                        :     :     :     +- Scan parquet  (3)
                        :     :     +- BroadcastExchange (9)
                        :     :        +- * Filter (8)
               

DataFrame[artist_id: string, user_id: string, event_id: string, genre: string, region: string, duration_sec: bigint, completed: boolean, device: string, event_date: date, month: int, plan: string, country: string, cpm: double, per_stream_rate: double, label_type: string, ad_revenue: double, stream_payout: double]

In [21]:
## Part 5: Write Summary Report (10 min)
# Create a final optimization report:
print("=" * 65)
print("OPTIMIZATION REPORT — StreamPulse Revenue Pipeline")
print("=" * 65)

print(f"""
Pipeline: Revenue Analytics (6 reports from joined data)

CONFIGURATION CHANGES:
  spark.sql.autoBroadcastJoinThreshold: -1 → 10MB
  spark.sql.shuffle.partitions: 200 → 8
  spark.sql.adaptive.enabled: false → (unchanged for testing)

CODE CHANGES:
  1. broadcast() on ad_rates (48 rows) and payouts (5K rows)
  2. .cache() on enriched DataFrame (built once, used 6 times)
  3. Column pruning on all source tables
  4. Single build_revenue() call instead of 6 separate calls

RESULTS:
  Baseline:  {baseline_total:.2f}s
  Optimized: {optimized_total:.2f}s
  Speedup:   {baseline_total/optimized_total:.1f}x

PLAN IMPROVEMENTS:
  - SortMergeJoin → BroadcastHashJoin (ad_rates, payouts)
  - 6 full recomputations → 1 computation + 5 cache reads
  - 200 shuffle partitions → 8 (matched to local cores)
  - ReadSchema reduced (column pruning)
""")


OPTIMIZATION REPORT — StreamPulse Revenue Pipeline

Pipeline: Revenue Analytics (6 reports from joined data)

CONFIGURATION CHANGES:
  spark.sql.autoBroadcastJoinThreshold: -1 → 10MB
  spark.sql.shuffle.partitions: 200 → 8
  spark.sql.adaptive.enabled: false → (unchanged for testing)

CODE CHANGES:
  1. broadcast() on ad_rates (48 rows) and payouts (5K rows)
  2. .cache() on enriched DataFrame (built once, used 6 times)
  3. Column pruning on all source tables
  4. Single build_revenue() call instead of 6 separate calls

RESULTS:
  Baseline:  200.87s
  Optimized: 18.30s
  Speedup:   11.0x

PLAN IMPROVEMENTS:
  - SortMergeJoin → BroadcastHashJoin (ad_rates, payouts)
  - 6 full recomputations → 1 computation + 5 cache reads
  - 200 shuffle partitions → 8 (matched to local cores)
  - ReadSchema reduced (column pruning)

